In [9]:
import pandas as pd
import numpy as np
import re
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, LongType, StringType, ArrayType
from pyspark.sql.functions import explode, lower, col, collect_set, trim, regexp_replace

# --- CONFIGURABLE PARAMETERS ---
# Only one of these should be True at a time!
profile = 'healthy'  # Options: 'none', 'healthy', 'high_protein', 'high_fiber'
ingredients = ['banana', 'milk', 'sugar']  # e.g. ['banana', 'milk', 'sugar', 'potato', 'almonds']
kcal_min = None   # e.g. 100
kcal_max = None   # e.g. 300
dislikes = []     # e.g. ['onion', 'garlic']
top_n = 10        # Number of recipe ideas to return

In [10]:
# --- LOAD DATA ---
spark = SparkSession.builder.appName('RecipeIdeas').getOrCreate()
# Ingredients
ingredients_schema = StructType([
    StructField('fdc_id', LongType(), False),
    StructField('description', StringType(), True),
    StructField('all_ingredients', ArrayType(StringType()), True)
])
df_ing = spark.read.schema(ingredients_schema).parquet(
    '../output/ingredients_nutrional_profiles/ingredients_nutrional_profiles.parquet/part-00000-33e03f78-d7e8-41fa-a09f-4a8c4605dc35-c000.snappy.parquet'
)
# Nutrients
df_nutri = spark.read.parquet('../output/nutritional_profiles/part-00000-006c6ee8-befb-47d8-8451-e7dae4de0d2f-c000.snappy.parquet')
# Merge
df = df_ing.join(df_nutri, df_ing.fdc_id == df_nutri.fdc_id, 'inner')
df = df.select(df_ing.fdc_id, df_ing.description, 'all_ingredients', *[c for c in df_nutri.columns if c != 'fdc_id'])
foods = df.toPandas()

In [11]:
# --- INGREDIENT NORMALIZATION ---
def normalize_ingredient(ing):
    # Remove punctuation, lowercase, trim
    return re.sub(r'[\(\)\"\*\[\]\{\}\.,:;\'\-&]', '', ing.lower().strip())
foods['ingredients_set'] = foods['all_ingredients'].apply(lambda lst: set(normalize_ingredient(i) for i in lst) if isinstance(lst, list) else set())

In [ ]:
# --- FILTERING FUNCTION ---
def filter_foods(foods, ingredients, dislikes, kcal_min, kcal_max):
    # Filter by dislikes
    if dislikes:
        dislikes_set = set(normalize_ingredient(d) for d in dislikes)
        foods = foods[~foods['ingredients_set'].apply(lambda s: any(d in s for d in dislikes_set))]
    # Filter by kcal
    if kcal_min is not None:
        foods = foods[foods['energy'].notnull() & (foods['energy'] >= kcal_min)]
    if kcal_max is not None:
        foods = foods[foods['energy'].notnull() & (foods['energy'] <= kcal_max)]
    # Filter by available ingredients: at least min(3, len(ingredients)) matches
    if ingredients:
        ing_set = set(normalize_ingredient(i) for i in ingredients)
        min_match = min(3, len(ing_set))
        foods = foods[foods['ingredients_set'].apply(lambda s: len(ing_set & s) >= min_match)]
    return foods.copy()

In [13]:
# --- SCORING FUNCTION ---
# Default weights for each profile
PROFILE_WEIGHTS = {
    'none':     {'fiber': 0,   'protein': 0,   'sugars': 0,   'total_fat': 0,   'energy': 0},
    'healthy':  {'fiber': 1.5, 'protein': 1.2, 'sugars': -1.5, 'total_fat': -0.8, 'energy': -0.01},
    'high_protein': {'fiber': 0, 'protein': 2.0, 'sugars': 0, 'total_fat': 0, 'energy': 0},
    'high_fiber':   {'fiber': 2.0, 'protein': 0, 'sugars': 0, 'total_fat': 0, 'energy': 0}
}

def score_food(row, ing_set, profile):
    score = 0.0
    # Ingredient match
    score += 2.0 * len(ing_set & row['ingredients_set'])
    # Nutrient weights by profile
    weights = PROFILE_WEIGHTS.get(profile, PROFILE_WEIGHTS['none'])
    if pd.notnull(row.get('fiber')): score += weights['fiber'] * row['fiber']
    if pd.notnull(row.get('protein')): score += weights['protein'] * row['protein']
    if pd.notnull(row.get('sugars')): score += weights['sugars'] * row['sugars']
    if pd.notnull(row.get('total_fat')): score += weights['total_fat'] * row['total_fat']
    if pd.notnull(row.get('energy')): score += weights['energy'] * row['energy']
    return score

In [14]:
# --- MAIN RECIPE SUGGESTION LOGIC ---
assert profile in ['none', 'healthy', 'high_protein', 'high_fiber'], "Profile must be one of: 'none', 'healthy', 'high_protein', 'high_fiber'"
ing_set = set(normalize_ingredient(i) for i in ingredients)
filtered = filter_foods(foods, ingredients, dislikes, kcal_min, kcal_max)
if filtered.shape[0] == 0:
    print('No recipe ideas found for your configuration.')
else:
    filtered['recipe_score'] = filtered.apply(lambda row: score_food(row, ing_set, profile), axis=1)
    filtered = filtered.sort_values('recipe_score', ascending=False)
    print('Top recipe ideas:')
    display(filtered[['description', 'ingredients_set', 'energy', 'fiber', 'protein', 'sugars', 'total_fat', 'recipe_score']].head(top_n))

Top recipe ideas:


,description,ingredients_set,energy,fiber,protein,sugars,total_fat,recipe_score
3227,"MM MANIA, FIBER BREADSTICKS, GARLIC & SEASAME,...","{salt, modified wheat starch resistant starch,...",212.0,38.9,10.60,3.53,8.83,58.591
2138,TURKEY JERKY,"{turkey breast, salt, cane molasses, soybeans,...",292.0,0.0,50.00,12.50,0.00,40.330
55,"ROASTED GARLIC CRISP-ROASTED CHICKPEAS, ROASTE...","{salt, spice, maltodextrin, natural flavor, ye...",455.0,18.2,22.73,4.55,13.64,34.289
1911,"REAL BACON BITS, SMOKE","{salt, bacon cured with water, sodium nitrite ...",357.0,NaN,42.86,NaN,21.43,32.718
248,TINY SHRIMP,"{salt, sodium metabisulfite preservative, citr...",125.0,0.0,26.79,0.00,1.79,31.466
1874,"GOLDEN WHEAT HAMBURGER BUNS, GOLDEN WHEAT","{erythritol, guar gum, xanthan gum, ingredient...",219.0,15.6,17.19,3.12,12.50,29.158
665,ROTISSERIE STYLE SHREDDED CHICKEN BREAST WITH ...,"{salt, vinegar, chicken base roasted chicken s...",141.0,0.0,25.88,1.18,2.94,27.524
2824,"GRASS-FED WHEY PROTEIN BARS, BROWNIE CRISP","{vanilla, vegetable glycerin, organic chocolat...",380.0,14.0,28.00,10.00,14.00,26.600
3832,Hardwood Smoked PREM Sugar Cured Center Cut Ba...,"{salt, cured with water, sodium phosphate, sod...",429.0,0.0,42.86,NaN,28.57,26.286
1108,CRUMBLED BACON BITS,"{salt, bacon cured with water, sodium nitrite ...",429.0,0.0,42.86,0.00,28.57,26.286


---
**How it works:**
- Enter your available ingredients, dislikes, kcal range, and nutrition preferences at the top.
- The notebook will suggest foods/recipes that match your configuration, prioritizing ingredient overlap and nutrition profile.
- You can easily adjust the scoring logic for your needs.